- Libraries needed for all feature extraction

In [11]:
import os
import glob
import numpy as np
import pandas as pd
import math
from scipy import stats, signal
import pywt
from scipy.io import loadmat
import concurrent.futures
from scipy.stats import skew, kurtosis
from scipy.signal import welch
from scipy.spatial.distance import pdist, squareform


### 1. Time Domain Features

In [12]:
# ----- Helper functions for time-domain features -----
def rms(signal):
    return np.sqrt(np.mean(np.square(signal)))

def zero_crossings(signal):
    return np.sum(np.diff(np.sign(signal)) != 0)

def line_length(signal):
    return np.sum(np.abs(np.diff(signal)))

def extract_time_features(signal):
    # Remove NaNs if any
    sig = np.array(signal)[~np.isnan(signal)]
    features = {}
    features['mean'] = np.mean(sig)
    features['std'] = np.std(sig)
    features['variance'] = np.var(sig)
    features['min'] = np.min(sig)
    features['max'] = np.max(sig)
    features['ptp'] = features['max'] - features['min']
    features['skew'] = skew(sig)
    features['kurtosis'] = kurtosis(sig)
    features['rms'] = rms(sig)
    features['zero_crossings'] = zero_crossings(sig)
    features['line_length'] = line_length(sig)
    return features

def extract_features_from_file(file_path):
    mat_data = loadmat(file_path)
    # Assume that the EEG variable is the first key that doesn't start with '__'
    keys = [k for k in mat_data.keys() if not k.startswith('__')]
    if not keys:
        return None
    data = mat_data[keys[0]]
    # Ensure data is (time x channels). If shape[0] is less than shape[1], assume data is transposed.
    if data.shape[0] < data.shape[1]:
        data = data.T

    file_features = {}
    # Extract features for each channel
    n_channels = data.shape[1]
    for ch in range(n_channels):
        ch_feats = extract_time_features(data[:, ch])
        # Prefix feature names with channel identifier
        for feat, val in ch_feats.items():
            file_features[f"ch{ch+1}_{feat}"] = val
    return file_features

# ----- Main extraction loop -----
# Change 'ArtifactFreeData' to the correct folder path on your system
base_dir = os.getcwd()  # or set it to your project directory path
data_dir = os.path.join(base_dir, "ArtifactFreeData")

# List all sub-folders (e.g., ADHD_part1, ADHD_part2, etc.)
subfolders = [f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))]
all_features = []

for folder in subfolders:
    folder_path = os.path.join(data_dir, folder)
    mat_files = glob.glob(os.path.join(folder_path, "*.mat"))
    for file in mat_files:
        feats = extract_features_from_file(file)
        if feats is None:
            continue
        feats['folder'] = folder
        feats['filename'] = os.path.basename(file)
        all_features.append(feats)

df_features = pd.DataFrame(all_features)
print("Time-Domain Feature Extraction Results (first 5 rows):")
print(df_features.head())

output_csv = "time_features.csv"
df_features.to_csv(output_csv, index=False)
print(f"Saved time-domain features to {output_csv}")


Time-Domain Feature Extraction Results (first 5 rows):
   ch1_mean   ch1_std  ch1_variance   ch1_min   ch1_max   ch1_ptp  ch1_skew  \
0 -0.018831  0.830215      0.689257 -2.995718  2.984256  5.979974 -0.012892   
1 -0.021761  0.867874      0.753206 -2.966287  2.994925  5.961212  0.035477   
2  0.008345  0.968048      0.937118 -2.976503  2.996632  5.973136  0.013677   
3 -0.090265  0.825986      0.682253 -2.947153  2.999776  5.946929  0.297300   
4 -0.018229  0.924432      0.854574 -2.995018  2.993448  5.988466 -0.030589   

   ch1_kurtosis   ch1_rms  ch1_zero_crossings  ...  ch19_min  ch19_max  \
0      0.926383  0.830429                2979  ... -2.975962  2.973540   
1      0.431210  0.868147                2271  ... -2.993762  2.968944   
2     -0.067901  0.968084                4526  ... -2.998162  2.992124   
3      1.904433  0.830904                1662  ... -2.978844  2.976507   
4      0.135715  0.924611                1793  ... -2.999426  2.996403   

   ch19_ptp  ch19_skew  c

### 2. Frequency Domain Features

In [13]:
# ----- Helper functions for frequency-domain features -----

def spectral_entropy(psd, freqs):
    psd_norm = psd / (np.sum(psd) + 1e-12)
    return -np.sum(psd_norm * np.log2(psd_norm + 1e-12))

def band_power(psd, freqs, fmin, fmax):
    idx = np.logical_and(freqs >= fmin, freqs < fmax)
    return np.trapz(psd[idx], freqs[idx])

def spectral_edge_frequency(psd, freqs, edge=0.9):
    total_power = np.sum(psd)
    cum_power = np.cumsum(psd)
    idx = np.where(cum_power >= edge * total_power)[0]
    if len(idx) > 0:
        return freqs[idx[0]]
    else:
        return np.nan

def psd_slope_r2(psd, freqs):
    valid = freqs > 0
    x = np.log10(freqs[valid])
    y = np.log10(psd[valid])
    slope, intercept = np.polyfit(x, y, 1)
    y_pred = slope * x + intercept
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan
    return slope, r2

def extract_freq_features(signal, fs=128):
    sig = np.array(signal)[~np.isnan(signal)]
    nperseg = min(256, len(sig))
    freqs, psd = welch(sig, fs=fs, nperseg=nperseg)
    
    features = {}
    bands = {
        'delta': (0.5, 4),
        'theta': (4, 8),
        'alpha': (8, 13),
        'beta':  (13, 30),
        'gamma': (30, 45)
    }
    for band, (fmin, fmax) in bands.items():
        features[f'{band}_power'] = band_power(psd, freqs, fmin, fmax)
    features['spectral_entropy'] = spectral_entropy(psd, freqs)
    features['spectral_edge_freq'] = spectral_edge_frequency(psd, freqs, edge=0.9)
    features['psd_slope'], features['psd_r2'] = psd_slope_r2(psd, freqs)
    
    return features

def extract_freq_features_from_file(file_path, fs=128):
    mat_data = loadmat(file_path)
    keys = [k for k in mat_data.keys() if not k.startswith('__')]
    if not keys:
        return None
    data = mat_data[keys[0]]
    if data.shape[0] < data.shape[1]:
        data = data.T
    file_features = {}
    n_channels = data.shape[1]
    for ch in range(n_channels):
        ch_feats = extract_freq_features(data[:, ch], fs=fs)
        for feat, val in ch_feats.items():
            file_features[f"ch{ch+1}_{feat}"] = val
    return file_features

# ----- Main extraction loop for frequency-domain features -----
base_dir = os.getcwd()  # Set this to your project directory if needed
data_dir = os.path.join(base_dir, "ArtifactFreeData")
subfolders = [f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))]
all_freq_features = []

for folder in subfolders:
    folder_path = os.path.join(data_dir, folder)
    mat_files = glob.glob(os.path.join(folder_path, "*.mat"))
    for file in mat_files:
        feats = extract_freq_features_from_file(file)
        if feats is None:
            continue
        feats['folder'] = folder
        feats['filename'] = os.path.basename(file)
        all_freq_features.append(feats)

df_freq_features = pd.DataFrame(all_freq_features)
print("Frequency-Domain Feature Extraction Results (first 5 rows):")
print(df_freq_features.head())

# Save the frequency-domain features DataFrame to a CSV file
output_csv = "freq_features.csv"
df_freq_features.to_csv(output_csv, index=False)
print(f"Saved frequency-domain features to {output_csv}")



Frequency-Domain Feature Extraction Results (first 5 rows):
   ch1_delta_power  ch1_theta_power  ch1_alpha_power  ch1_beta_power  \
0         0.259170         0.150521         0.063900        0.103412   
1         0.278744         0.168068         0.107693        0.105048   
2         0.358652         0.188367         0.087327        0.066271   
3         0.418563         0.135493         0.035885        0.028369   
4         0.395681         0.145126         0.065000        0.153102   

   ch1_gamma_power  ch1_spectral_entropy  ch1_spectral_edge_freq  \
0         0.016153              5.344984                    24.0   
1         0.020861              5.318916                    21.5   
2         0.009883              4.885338                    50.0   
3         0.004007              4.247608                     8.5   
4         0.024853              5.120428                    21.5   

   ch1_psd_slope  ch1_psd_r2  ch2_delta_power  ...  ch19_theta_power  \
0      -2.140718    0.6554

### 3. Non linear/Complexity features

In [8]:
import os
import glob
import numpy as np
import pandas as pd
import scipy.io
import antropy as ant

# Define a function to compute Hjorth parameters on a NaN-free signal
def compute_hjorth(signal):
    # Remove any NaN values
    signal = signal[~np.isnan(signal)]
    if len(signal) == 0:
        return np.nan, np.nan, np.nan
    activity = np.nanvar(signal)  # Using nanvar to be safe
    if activity == 0:
        return activity, np.nan, np.nan
    first_deriv = np.diff(signal)
    mobility = np.sqrt(np.nanvar(first_deriv) / activity)
    var_first = np.nanvar(first_deriv)
    if var_first == 0:
        complexity = np.nan
    else:
        second_deriv = np.diff(first_deriv)
        mobility2 = np.sqrt(np.nanvar(second_deriv) / var_first)
        complexity = mobility2 / mobility if mobility != 0 else np.nan
    return activity, mobility, complexity

# Function to compute non-linear features for a single signal (after removing NaNs)
def compute_nonlinear_features(signal):
    # Remove NaNs from the signal before processing
    clean_signal = signal[~np.isnan(signal)]
    features = {}
    if len(clean_signal) == 0:
        features['approx_entropy'] = np.nan
        features['higuchi_fd'] = np.nan
        features['hjorth_mobility'] = np.nan
        features['hjorth_complexity'] = np.nan
        return features
    try:
        features['approx_entropy'] = ant.app_entropy(clean_signal)
    except Exception:
        features['approx_entropy'] = np.nan
    # Removed Hurst exponent calculation here
    try:
        features['higuchi_fd'] = ant.higuchi_fd(clean_signal)
    except Exception:
        features['higuchi_fd'] = np.nan
    try:
        _, mob, comp = compute_hjorth(clean_signal)
        features['hjorth_mobility'] = mob
        features['hjorth_complexity'] = comp
    except Exception:
        features['hjorth_mobility'] = np.nan
        features['hjorth_complexity'] = np.nan
    return features

# Function to process a single .mat file and extract non-linear features for each channel
def process_file(filepath):
    try:
        mat = scipy.io.loadmat(filepath)
        # Get keys that don't start with '__'
        keys = [k for k in mat.keys() if not k.startswith('__')]
        if not keys:
            return None
        data = mat[keys[0]]  # Assume shape is (samples, channels)
        file_features = {'folder': os.path.basename(os.path.dirname(filepath)),
                         'filename': os.path.basename(filepath)}
        num_channels = data.shape[1]
        for ch in range(num_channels):
            signal = data[:, ch].flatten()
            feat = compute_nonlinear_features(signal)
            for key, value in feat.items():
                file_features[f'ch{ch+1}_{key}'] = value
        return file_features
    except Exception as e:
        print(f"Error processing {filepath}: {e}")
        return None

# Define folders to process
folders = [
    "/home/niranjanrao07/ML-project/ArtifactFreeData/ADHD_part1",
    "/home/niranjanrao07/ML-project/ArtifactFreeData/ADHD_part2",
    "/home/niranjanrao07/ML-project/ArtifactFreeData/Control_part1",
    "/home/niranjanrao07/ML-project/ArtifactFreeData/Control_part2"
]

all_results = []
for folder in folders:
    file_list = glob.glob(os.path.join(folder, "*.mat"))
    for f in file_list:
        result = process_file(f)
        if result is not None:
            all_results.append(result)

df_nonlinear = pd.DataFrame(all_results)
print("Non-linear/Complexity Feature Extraction Results (first 5 rows):")
print(df_nonlinear.head())

# Save the results to a CSV file
output_csv = "nonlinear_features_all.csv"
df_nonlinear.to_csv(output_csv, index=False)
print(f"Saved nonlinear features to {output_csv}")


Non-linear/Complexity Feature Extraction Results (first 5 rows):
       folder  filename  ch1_approx_entropy  ch1_higuchi_fd  \
0  ADHD_part1  v39p.mat            1.742218        1.641895   
1  ADHD_part1  v21p.mat            1.311857        1.496660   
2  ADHD_part1  v19p.mat            1.377374        1.572209   
3  ADHD_part1  v12p.mat            1.357952        1.593953   
4  ADHD_part1  v40p.mat            1.340236        1.566656   

   ch1_hjorth_mobility  ch1_hjorth_complexity  ch2_approx_entropy  \
0             0.609372               2.391702            1.250830   
1             0.407174               3.063522            1.392746   
2             0.455142               2.474525            1.207463   
3             0.484203               3.392956            1.074351   
4             0.405501               3.082453            1.674040   

   ch2_higuchi_fd  ch2_hjorth_mobility  ch2_hjorth_complexity  ...  \
0        1.489283             0.356088               3.548438  ...   
1

In [1]:
import pandas as pd

# 1. Load the merged features
df = pd.read_csv("merged_features.csv")

# 2. Derive binary label: ADHD=1, Control=0
df["has_adhd"] = df["folder"].str.contains("ADHD", case=False).astype(int)

# 3. Optional: Inspect the first few rows
print(df.head())

# 4. Save the labeled dataset for sharing
output_path = "merged_features_labeled.csv"
df.to_csv(output_path, index=False)
print(f"Saved labeled data to {output_path}")


   ch1_mean   ch1_std  ch1_variance   ch1_min   ch1_max   ch1_ptp  ch1_skew  \
0 -0.018831  0.830215      0.689257 -2.995718  2.984256  5.979974 -0.012892   
1 -0.021761  0.867874      0.753206 -2.966287  2.994925  5.961212  0.035477   
2  0.008345  0.968048      0.937118 -2.976503  2.996632  5.973136  0.013677   
3 -0.090265  0.825986      0.682253 -2.947153  2.999776  5.946929  0.297300   
4 -0.018229  0.924432      0.854574 -2.995018  2.993448  5.988466 -0.030589   

   ch1_kurtosis   ch1_rms  ch1_zero_crossings  ...  ch17_hjorth_complexity  \
0      0.926383  0.830429                2979  ...                2.231082   
1      0.431210  0.868147                2271  ...                2.409458   
2     -0.067901  0.968084                4526  ...                2.344311   
3      1.904433  0.830904                1662  ...                3.031723   
4      0.135715  0.924611                1793  ...                2.697656   

   ch18_approx_entropy  ch18_higuchi_fd  ch18_hjorth_mob